<a href="https://colab.research.google.com/github/poggerssLL/Automatica---Grupo-2/blob/main/etapa-01-logica/06%20Quantificadores%20e%20Predicados%20em%20Redes%20de%20Sensores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 06 - Notebook: Lógica de Predicados e Quantificadores em Redes de Sensores Industriais

## 1. Fundamentos Matemáticos: Lógica de Primeira Ordem (FOL) na Automação

Enquanto a Lógica Proposicional manipula proposições atômicas indivisíveis ($p, q, r$), a **Lógica de Predicados (Lógica de Primeira Ordem - FOL)** permite parametrizar propriedades sobre domínios e conjuntos finitos de instrumentos e atuadores distribuídos na planta industrial:

$$\langle \mathcal{U}, \mathcal{P}, \mathcal{Q} \rangle$$

Onde:
1. **Universo de Discurso ($\mathcal{U}$):** Conjunto não vazio de elementos da planta (sensores capacitivos de copos, atuadores pneumáticos, etc.).
2. **Predicado $P(x)$:** Função booleana $P: \mathcal{U} \rightarrow \{0, 1\}$ que avalia se o elemento $x \in \mathcal{U}$ satisfaz uma dada propriedade operacional ou de segurança.
3. **Quantificadores Lógicos ($\mathcal{Q}$):**
   * **Quantificador Universal ($\forall x \in \mathcal{U}, \; P(x)$):** Afirma que a propriedade $P(x)$ é estritamente verdadeira para **todos** os elementos do conjunto.
     $$\forall x \in \mathcal{U} \, P(x) \equiv \bigwedge_{i=1}^n P(x_i) = P(x_1) \land P(x_2) \land \dots \land P(x_n)$$
   * **Quantificador Existencial ($\exists x \in \mathcal{U}, \; P(x)$):** Afirma que existe **ao menos um** elemento em $\mathcal{U}$ que satisfaz $P(x)$.
     $$\exists x \in \mathcal{U} \, P(x) \equiv \bigvee_{i=1}^n P(x_i) = P(x_1) \lor P(x_2) \lor \dots \lor P(x_n)$$

### Leis de De Morgan para Quantificadores
$$\neg \left( \forall x \in \mathcal{U} \, P(x) \right) \equiv \exists x \in \mathcal{U} \, \neg P(x)$$
$$\neg \left( \exists x \in \mathcal{U} \, P(x) \right) \equiv \forall x \in \mathcal{U} \, \neg P(x)$$

---

## 2. Aplicação na Máquina de Envasamento de Copos Plásticos (UNIFEI)

Neste notebook implementamos:
1. O motor genérico de primeira ordem `MotorPredicadosFOL` com operadores `forall` e `exists`.
2. A modelagem orientada a objetos da rede de sensores capacitivos ($s_1 \dots s_5$) e dos 7 pares de fins de curso magnéticos ($c_{1a}/c_{1r} \dots c_{9a}/c_{9r}$).
3. A verificação do permissivo global de repouso da mesa giratória e detecção de incoerência de sensores.
4. A demonstração formal computacional das **Leis de De Morgan para Quantificadores**.
5. O módulo injetor de falhas dinâmicas para simulação em tempo real no SCADA.


In [1]:
from typing import Callable, Dict, List, Any, Tuple
from dataclasses import dataclass

def formatar_tabela(dados: List[Dict[str, Any]]) -> str:
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

# ==============================================================================
# MODELAGEM DE ESTRUTURAS DE DADOS DA PLANTA INDUSTRIAL
# ==============================================================================

@dataclass
class SensorCapacitivo:
    tag: str
    setor: str
    estacao: int
    copo_presente: bool

@dataclass
class CilindroPneumatico:
    nome: str
    setor: str
    tag_avanco: str
    tag_recuo: str
    avanco_detectado: bool
    recuo_detectado: bool
    posicao_repouso_avancada: bool = False

@dataclass
class InstrumentoAnalitico:
    tag: str
    variavel: str
    valor: float
    limite_min: float
    limite_max: float
    unidade: str

# ==============================================================================
# MOTOR DE LÓGICA DE PRIMEIRA ORDEM (FOL ENGINE)
# ==============================================================================

class MotorPredicadosFOL:
    """Motor genérico para avaliação de quantificadores lógicos sobre domínios finitos."""

    @staticmethod
    def forall(universo: List[Any], predicado: Callable[[Any], bool]) -> Tuple[bool, List[Any]]:
        """
        Avalia o Quantificador Universal: PARA TODO x in U, P(x).
        Retorna (Resultado Booleano, Lista de Contraexemplos).
        """
        contraexemplos = [x for x in universo if not predicado(x)]
        return len(contraexemplos) == 0, contraexemplos

    @staticmethod
    def exists(universo: List[Any], predicado: Callable[[Any], bool]) -> Tuple[bool, List[Any]]:
        """
        Avalia o Quantificador Existencial: EXISTE x in U tal que P(x).
        Retorna (Resultado Booleano, Lista de Exemplos que satisfazem).
        """
        exemplos = [x for x in universo if predicado(x)]
        return len(exemplos) > 0, exemplos

# ==============================================================================
# PREDICADOS ESPECÍFICOS DA MÁQUINA DE ENVASE
# ==============================================================================

def pred_copo_presente(s: SensorCapacitivo) -> bool:
    return s.copo_presente

def pred_sensor_incoerente(c: CilindroPneumatico) -> bool:
    """Retorna True se ambos os fins de curso (avanço e recuo) estiverem ativos simultaneamente."""
    return c.avanco_detectado and c.recuo_detectado

def pred_cilindro_em_repouso(c: CilindroPneumatico) -> bool:
    """Retorna True se o atuador está na posição segura de repouso autorizada para o giro."""
    if c.posicao_repouso_avancada:
        return c.avanco_detectado and not c.recuo_detectado
    else:
        return c.recuo_detectado and not c.avanco_detectado

def pred_parametro_conforme(inst: InstrumentoAnalitico) -> bool:
    return inst.limite_min <= inst.valor <= inst.limite_max

# ==============================================================================
# SUPERVISÓRIO SCADA BASEADO EM PREDICADOS
# ==============================================================================

class SupervisorioSCADAEnvasadora:
    def __init__(self):
        self.fol = MotorPredicadosFOL()
        self.sensores_copo: List[SensorCapacitivo] = []
        self.cilindros: List[CilindroPneumatico] = []
        self.instrumentos: List[InstrumentoAnalitico] = []
        self.inicializar_planta_nominal()

    def inicializar_planta_nominal(self):
        # 5 Sensores Capacitivos (Setores 100 a 500)
        self.sensores_copo = [
            SensorCapacitivo(tag="ZS-102", setor="Setor 100 (Dispensa)", estacao=1, copo_presente=True),
            SensorCapacitivo(tag="ZS-200", setor="Setor 200 (Envase)", estacao=2, copo_presente=True),
            SensorCapacitivo(tag="ZS-300", setor="Setor 300 (Tampa)", estacao=3, copo_presente=True),
            SensorCapacitivo(tag="ZS-400", setor="Setor 400 (Termosselagem)", estacao=4, copo_presente=True),
            SensorCapacitivo(tag="ZS-500", setor="Setor 500 (Ejeção)", estacao=5, copo_presente=True),
        ]

        # 7 Cilindros Pneumáticos de Dupla Ação
        self.cilindros = [
            CilindroPneumatico(nome="Cilindro A (Retentor Dispensa)", setor="Setor 100", tag_avanco="ZSC-101 (c1a)", tag_recuo="ZSO-101 (c1r)", avanco_detectado=True, recuo_detectado=False, posicao_repouso_avancada=True),
            CilindroPneumatico(nome="Cilindro C (Dosador 150ml)", setor="Setor 200", tag_avanco="ZSO-202 (c3a)", tag_recuo="ZSC-202 (c3r)", avanco_detectado=True, recuo_detectado=False, posicao_repouso_avancada=True),
            CilindroPneumatico(nome="Cilindro D (Giro Pick-and-Place)", setor="Setor 300", tag_avanco="ZSO-301 (c5a)", tag_recuo="ZSC-301 (c5r)", avanco_detectado=False, recuo_detectado=True, posicao_repouso_avancada=False),
            CilindroPneumatico(nome="Cilindro E (Vertical Pick-and-Place)", setor="Setor 300", tag_avanco="ZSO-302 (c6a)", tag_recuo="ZSC-302 (c6r)", avanco_detectado=False, recuo_detectado=True, posicao_repouso_avancada=False),
            CilindroPneumatico(nome="Cilindro F (Prensa Termosselagem)", setor="Setor 400", tag_avanco="ZSO-401 (c7a)", tag_recuo="ZSC-401 (c7r)", avanco_detectado=False, recuo_detectado=True, posicao_repouso_avancada=False),
            CilindroPneumatico(nome="Cilindro G (Elevador Ejeção)", setor="Setor 500", tag_avanco="ZSO-501 (c8a)", tag_recuo="ZSC-501 (c8r)", avanco_detectado=False, recuo_detectado=True, posicao_repouso_avancada=False),
            CilindroPneumatico(nome="Cilindro H (Extrator Ejeção)", setor="Setor 500", tag_avanco="ZSC-502 (c9a)", tag_recuo="ZSO-502 (c9r)", avanco_detectado=True, recuo_detectado=False, posicao_repouso_avancada=True),
        ]

        # Instrumentação Contínua
        self.instrumentos = [
            InstrumentoAnalitico(tag="TIT-401", variavel="Temperatura Cabeçote Prensa", valor=185.0, limite_min=180.0, limite_max=220.0, unidade="°C"),
            InstrumentoAnalitico(tag="PIT-301", variavel="Pressão Vácuo Ventosa", valor=-0.75, limite_min=-1.0, limite_max=-0.50, unidade="bar"),
        ]

    def status_rede_sensores(self) -> Dict[str, Any]:
        consistente, falhas = self.fol.forall(self.cilindros, lambda c: not pred_sensor_incoerente(c))
        repouso_ok, pendentes = self.fol.forall(self.cilindros, pred_cilindro_em_repouso)
        mesa_cheia, vazios = self.fol.forall(self.sensores_copo, pred_copo_presente)
        existe_copo, ocupados = self.fol.exists(self.sensores_copo, pred_copo_presente)
        qualidade_ok, desvios = self.fol.forall(self.instrumentos, pred_parametro_conforme)

        return {
            "Consistência Sensores (FORALL ~Incoerente)": consistente,
            "Prontidão Repouso (FORALL Repouso)": repouso_ok,
            "Mesa 100% Carregada (FORALL CopoPresente)": mesa_cheia,
            "Existe Algum Copo (EXISTS CopoPresente)": existe_copo,
            "Instrumentação Conforme (FORALL Conforme)": qualidade_ok,
            "Detalhes": {
                "falhas_sensores": [f"{c.nome} ({c.tag_avanco} & {c.tag_recuo})" for c in falhas],
                "pendencias_repouso": [c.nome for c in pendentes],
                "bercos_vazios": [s.setor for s in vazios],
                "desvios_processo": [f"{i.tag}: {i.valor}{i.unidade}" for i in desvios]
            }
        }

def demonstrar_de_morgan(universo: List[Any], predicado: Callable[[Any], bool], nome_teste: str) -> Dict[str, str]:
    # Lei 1: ~(forall x P(x)) == exists x ~P(x)
    forall_p, _ = MotorPredicadosFOL.forall(universo, predicado)
    not_forall_p = not forall_p
    exists_not_p, _ = MotorPredicadosFOL.exists(universo, lambda x: not predicado(x))
    lei1_ok = (not_forall_p == exists_not_p)

    # Lei 2: ~(exists x P(x)) == forall x ~P(x)
    exists_p, _ = MotorPredicadosFOL.exists(universo, predicado)
    not_exists_p = not exists_p
    forall_not_p, _ = MotorPredicadosFOL.forall(universo, lambda x: not predicado(x))
    lei2_ok = (not_exists_p == forall_not_p)

    return {
        "Propriedade Testada": nome_teste,
        "~FORALL P(x)": str(not_forall_p),
        "EXISTS ~P(x)": str(exists_not_p),
        "Lei 1 De Morgan": "VALIDADA [OK]" if lei1_ok else "FALHOU",
        "~EXISTS P(x)": str(not_exists_p),
        "FORALL ~P(x)": str(forall_not_p),
        "Lei 2 De Morgan": "VALIDADA [OK]" if lei2_ok else "FALHOU"
    }

# ==============================================================================
# BATERIA DE TESTES E VALIDAÇÃO COMPUTACIONAL
# ==============================================================================

scada = SupervisorioSCADAEnvasadora()

print("=======================================================================")
print("CENÁRIO 1: PLANTA NOMINAL (CONDIÇÃO SEGURA DE REPOUSO)")
print("=======================================================================")
res1 = scada.status_rede_sensores()
tabela1 = [{"Verificação Lógica": k, "Resultado": str(v)} for k, v in res1.items() if k != "Detalhes"]
print(formatar_tabela(tabela1))

print("\n=======================================================================")
print("DEMONSTRAÇÃO COMPUTACIONAL DAS LEIS DE DE MORGAN PARA QUANTIFICADORES")
print("=======================================================================")
tabela_dm = [
    demonstrar_de_morgan(scada.cilindros, pred_cilindro_em_repouso, "Repouso Seguro dos Cilindros"),
    demonstrar_de_morgan(scada.sensores_copo, pred_copo_presente, "Presença de Copo nas Estações"),
    demonstrar_de_morgan(scada.cilindros, lambda c: not pred_sensor_incoerente(c), "Consistência dos Fins de Curso")
]
print(formatar_tabela(tabela_dm))

print("\n=======================================================================")
print("CENÁRIO 2: INJEÇÃO DE FALHA (CURTO-CIRCUITO NO SENSOR DA PRENSA CILINDRO F)")
print("=======================================================================")
# Injeta falha no cilindro F (índice 4): avanço e recuo simultâneos
scada.cilindros[4].avanco_detectado = True
scada.cilindros[4].recuo_detectado = True

res2 = scada.status_rede_sensores()
tabela2 = [{"Verificação Lógica": k, "Resultado": str(v)} for k, v in res2.items() if k != "Detalhes"]
print(formatar_tabela(tabela2))
print("\n[ALERTA DE FALHA EM CAMPO] Detectado:", res2["Detalhes"]["falhas_sensores"])

assert res2["Consistência Sensores (FORALL ~Incoerente)"] is False
assert len(res2["Detalhes"]["falhas_sensores"]) == 1
assert "Cilindro F (Prensa Termosselagem)" in res2["Detalhes"]["falhas_sensores"][0]

print("\n[OK] Laboratório 06 validado com 100% de sucesso formal e operacional!")


CENÁRIO 1: PLANTA NOMINAL (CONDIÇÃO SEGURA DE REPOUSO)
Verificação Lógica                         | Resultado
-------------------------------------------+----------
Consistência Sensores (FORALL ~Incoerente) | True     
Prontidão Repouso (FORALL Repouso)         | True     
Mesa 100% Carregada (FORALL CopoPresente)  | True     
Existe Algum Copo (EXISTS CopoPresente)    | True     
Instrumentação Conforme (FORALL Conforme)  | True     

DEMONSTRAÇÃO COMPUTACIONAL DAS LEIS DE DE MORGAN PARA QUANTIFICADORES
Propriedade Testada            | ~FORALL P(x) | EXISTS ~P(x) | Lei 1 De Morgan | ~EXISTS P(x) | FORALL ~P(x) | Lei 2 De Morgan
-------------------------------+--------------+--------------+-----------------+--------------+--------------+----------------
Repouso Seguro dos Cilindros   | False        | False        | VALIDADA [OK]   | False        | False        | VALIDADA [OK]  
Presença de Copo nas Estações  | False        | False        | VALIDADA [OK]   | False        | False       